# Project 15 — Final comparison
Run `final_data.ipynb` first. All interpretation is in `final_checkpoint.pdf`.

In [1]:
import os

os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["USE_TF"] = "0"

import platform
import random
import re
import time
import unicodedata
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from huggingface_hub import hf_hub_download
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    average_precision_score,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from transformers import AutoModel, AutoTokenizer

OUTPUT_DIR = Path.cwd()
RANDOM_SEED = 15
MODEL_NAME = "microsoft/deberta-v3-base"
MODEL_REVISION = "8ccc9b6f36199bec6961081d44eb72fb3f7353f3"
MAX_TOKENS = 128
BATCH_SIZE = 8
INFERENCE_BATCH_SIZE = 32
VALIDATION_SHARE = 0.20
VALIDATION_FPR_LIMIT = 0.05
NEAR_DUPLICATE_SIMILARITY = 0.90
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.set_num_threads(min(4, os.cpu_count() or 1))

DATASET_SOURCES = {
    "deepset": {
        "repo": "deepset/prompt-injections",
        "revision": "4f61ecb038e9c3fb77e21034b22511b523772cdd",
        "files": {
            "train": "data/train-00000-of-00001-9564e8b05b4757ab.parquet",
            "test": "data/test-00000-of-00001-701d16158af87368.parquet",
        },
    },
    "safeguard": {
        "repo": "xTRam1/safe-guard-prompt-injection",
        "revision": "a3a877d608f37b7d20d9945671902df895ecdb46",
        "files": {
            "train": "data/train-00000-of-00001.parquet",
            "test": "data/test-00000-of-00001.parquet",
        },
    },
}
SAFEGUARD_SAMPLE_SIZES = {"train": 800, "test": 200}

hardware = platform.machine() + f" CPU; {os.cpu_count()} logical cores"
if DEVICE == "cuda":
    hardware = torch.cuda.get_device_name(0)
print("Python:", platform.python_version())
print("Hardware:", hardware)

Python: 3.13.16
Hardware: NVIDIA L4


In [2]:
dataset_frames = []

for dataset_name, source in DATASET_SOURCES.items():
    for publisher_split, filename in source["files"].items():
        download_path = hf_hub_download(
            source["repo"],
            filename,
            repo_type="dataset",
            revision=source["revision"], token=False,
        )
        frame = pd.read_parquet(download_path)
        frame["source_index"] = np.arange(len(frame))

        if dataset_name == "safeguard":
            sample_indices, _ = train_test_split(
                np.arange(len(frame)),
                train_size=SAFEGUARD_SAMPLE_SIZES[publisher_split],
                stratify=frame["label"],
                random_state=RANDOM_SEED,
            )
            frame = frame.iloc[np.sort(sample_indices)].copy()

        frame["dataset"] = dataset_name
        frame["publisher_split"] = publisher_split
        frame["row_id"] = [
            f"{dataset_name}:{publisher_split}:{index}"
            for index in frame["source_index"]
        ]
        dataset_frames.append(frame)

loaded_prompts = pd.concat(dataset_frames, ignore_index=True)
print("Loaded rows:", len(loaded_prompts))


def normalize_text(text):
    normalized = unicodedata.normalize("NFKC", str(text)).casefold()
    return re.sub(r"\s+", " ", normalized).strip()


loaded_prompts["normalized"] = loaded_prompts["text"].map(normalize_text)
assert loaded_prompts["label"].isin([0, 1]).all()
assert loaded_prompts["text"].notna().all()
assert loaded_prompts["normalized"].ne("").all()

audit_records = []
for dataset_name, dataset in loaded_prompts.groupby("dataset"):
    audit_records.append(
        {
            "dataset": dataset_name,
            "rows": len(dataset),
            "source_columns": 2,
            "negative": int(dataset["label"].eq(0).sum()),
            "positive": int(dataset["label"].eq(1).sum()),
            "positive_pct": 100 * dataset["label"].eq(1).mean(),
            "missing_target": int(dataset["label"].isna().sum()),
            "missing_text": int(dataset["text"].isna().sum()),
            "blank_text": int(dataset["normalized"].eq("").sum()),
            "exact_excess_text": int(dataset.duplicated("text").sum()),
            "exact_excess_text_label": int(dataset.duplicated(["text", "label"]).sum()),
            "normalized_excess": int(dataset.duplicated("normalized").sum()),
        }
    )

audit_results = pd.DataFrame(audit_records)
display(audit_results)
print(
    "Entity and timestamp checks: unavailable; source files contain only text and label."
)

(…)-00000-of-00001-9564e8b05b4757ab.parquet:   0%|          | 0.00/40.3k [00:00<?, ?B/s]

(…)-00000-of-00001-701d16158af87368.parquet:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/1.99M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/497k [00:00<?, ?B/s]

Loaded rows: 1662


,dataset,rows,source_columns,negative,positive,positive_pct,missing_target,missing_text,blank_text,exact_excess_text,exact_excess_text_label,normalized_excess
0,deepset,662,2,399,263,39.728097,0,0,0,0,0,0
1,safeguard,1000,2,695,305,30.500000,0,0,0,3,3,3


Entity and timestamp checks: unavailable; source files contain only text and label.


In [3]:
duplicate_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=1,
    dtype=np.float32,
)
duplicate_features = duplicate_vectorizer.fit_transform(loaded_prompts["normalized"])
text_similarities = cosine_similarity(duplicate_features)

# The upper triangle counts each related pair once, without self-pairs.
related_pairs = np.triu(text_similarities >= NEAR_DUPLICATE_SIMILARITY, k=1)
left_indices, right_indices = np.where(related_pairs)
group_parents = np.arange(len(loaded_prompts))


def find_group_root(row_index):
    while group_parents[row_index] != row_index:
        group_parents[row_index] = group_parents[group_parents[row_index]]
        row_index = group_parents[row_index]
    return row_index


def merge_groups(first_index, second_index):
    first_root = find_group_root(first_index)
    second_root = find_group_root(second_index)
    if first_root != second_root:
        group_parents[second_root] = first_root


for first_index, second_index in zip(left_indices, right_indices):
    merge_groups(first_index, second_index)

for matching_indices in loaded_prompts.groupby("normalized").indices.values():
    for row_index in matching_indices[1:]:
        merge_groups(matching_indices[0], row_index)

loaded_prompts["group"] = [
    find_group_root(row_index) for row_index in range(len(loaded_prompts))
]

near_duplicate_records = []
for dataset_name in DATASET_SOURCES:
    both_in_dataset = (
        loaded_prompts["dataset"].iloc[left_indices].to_numpy() == dataset_name
    ) & (loaded_prompts["dataset"].iloc[right_indices].to_numpy() == dataset_name)
    dataset_pairs = list(
        zip(
            left_indices[both_in_dataset],
            right_indices[both_in_dataset],
        )
    )
    affected_rows = set()
    nonexact_pair_count = 0
    for first_index, second_index in dataset_pairs:
        affected_rows.update([first_index, second_index])
        if (
            loaded_prompts["text"].iloc[first_index]
            != loaded_prompts["text"].iloc[second_index]
        ):
            nonexact_pair_count += 1

    near_duplicate_records.append(
        {
            "dataset": dataset_name,
            "near_pairs": len(dataset_pairs),
            "nonexact_near_pairs": nonexact_pair_count,
            "rows_in_near_pairs": len(affected_rows),
        }
    )

near_duplicate_audit = pd.DataFrame(near_duplicate_records)
display(near_duplicate_audit)

,dataset,near_pairs,nonexact_near_pairs,rows_in_near_pairs
0,deepset,28,28,49
1,safeguard,17,14,24


In [4]:
loaded_prompts["split"] = loaded_prompts["publisher_split"]
for dataset_name in DATASET_SOURCES:
    publisher_training = loaded_prompts["dataset"].eq(dataset_name) & loaded_prompts[
        "publisher_split"
    ].eq("train")
    training_indices, validation_indices = train_test_split(
        loaded_prompts.index[publisher_training].to_numpy(),
        test_size=VALIDATION_SHARE,
        stratify=loaded_prompts.loc[publisher_training, "label"],
        random_state=RANDOM_SEED,
    )
    loaded_prompts.loc[training_indices, "split"] = "train"
    loaded_prompts.loc[validation_indices, "split"] = "validation"

# Preserve evaluation rows: test takes priority over validation, then training.
split_priority = {"train": 0, "validation": 1, "test": 2}
loaded_prompts["priority"] = loaded_prompts["split"].map(split_priority)
highest_group_priority = loaded_prompts.groupby("group")["priority"].transform("max")
survives_boundary_check = loaded_prompts["priority"].eq(highest_group_priority)

surviving_prompts = loaded_prompts.loc[survives_boundary_check].sort_values(
    ["dataset", "publisher_split", "source_index"]
)
retained_indices = surviving_prompts.drop_duplicates("normalized").index
removed_prompts = loaded_prompts.loc[
    ~loaded_prompts.index.isin(retained_indices)
].copy()
retained_prompts = (
    loaded_prompts.loc[retained_indices].sort_index().reset_index(drop=True)
)

removal_counts = (
    removed_prompts.groupby(["dataset", "split"]).size().rename("removed").reset_index()
)
display(removal_counts)

training_prompts = retained_prompts[retained_prompts["split"].eq("train")]
leakage_counts = {}
for evaluation_split in ["test", "validation"]:
    evaluation_prompts = retained_prompts[
        retained_prompts["split"].eq(evaluation_split)
    ]
    for field, check_name in [
        ("text", "exact"),
        ("normalized", "normalized"),
        ("group", "near_groups"),
    ]:
        overlapping_rows = training_prompts[field].isin(set(evaluation_prompts[field]))
        leakage_counts[f"train_{check_name}_{evaluation_split}"] = int(
            overlapping_rows.sum()
        )

assert not any(leakage_counts.values())
test_groups = set(retained_prompts.loc[retained_prompts["split"].eq("test"), "group"])
validation_groups = set(
    retained_prompts.loc[retained_prompts["split"].eq("validation"), "group"]
)
assert test_groups.isdisjoint(validation_groups)

split_counts = (
    retained_prompts.groupby(["dataset", "split"])
    .agg(
        rows=("label", "size"),
        positive=("label", "sum"),
    )
    .reset_index()
)
split_counts["negative"] = split_counts["rows"] - split_counts["positive"]
split_counts["positive_pct"] = 100 * split_counts["positive"] / split_counts["rows"]

retained_prompts[["row_id", "dataset", "split", "label"]].to_csv(
    "split.csv", index=False
)
display(split_counts)
print("Training overlap counts:", leakage_counts)

,dataset,split,removed
0,deepset,train,11
1,deepset,validation,1
2,safeguard,test,1
3,safeguard,train,9


,dataset,split,rows,positive,negative,positive_pct
0,deepset,test,116,60,56,51.724138
1,deepset,train,425,158,267,37.176471
2,deepset,validation,109,40,69,36.697248
3,safeguard,test,199,63,136,31.658291
4,safeguard,train,631,192,439,30.427892
5,safeguard,validation,160,48,112,30.000000


Training overlap counts: {'train_exact_test': 0, 'train_normalized_test': 0, 'train_near_groups_test': 0, 'train_exact_validation': 0, 'train_normalized_validation': 0, 'train_near_groups_validation': 0}


In [5]:
from peft import LoraConfig, get_peft_model
from sklearn.decomposition import PCA
import gc
import zlib

SEEDS = [15, 16, 17]
MAX_EPOCHS = 20
perturbations = pd.read_csv("perturbations.csv")
assert not perturbations.duplicated(["row_id", "column"]).any()
concept_pairs = pd.read_csv("concept_pairs.csv")

if Path("midsemester_scores.csv").exists():
    old_scores = pd.read_csv("midsemester_scores.csv")
else:
    old_scores = pd.read_csv("scores.csv")
    assert "test_set" in old_scores
    old_scores.to_csv("midsemester_scores.csv", index=False)

training_mask = retained_prompts.split.eq("train").to_numpy()
validation_mask = retained_prompts.split.eq("validation").to_numpy()
test_mask = retained_prompts.split.eq("test").to_numpy()
labels = retained_prompts.label.to_numpy()

# Training examples and labels are identical across all adaptation seeds.
print("Training / validation / test:", training_mask.sum(), validation_mask.sum(), test_mask.sum())
print("Representation drift decision: yes >= 0.05; no <= 0.01; otherwise undecided.")

# Keep model selection and operating-threshold calibration independent.
validation_indices = np.flatnonzero(validation_mask)
strata = (retained_prompts.loc[validation_mask, "dataset"] + ":" +
          retained_prompts.loc[validation_mask, "label"].astype(str))
selection_indices, calibration_indices = train_test_split(
    validation_indices, test_size=0.5, stratify=strata, random_state=15)
calibration_groups = set(retained_prompts.loc[calibration_indices, "group"])
move_to_calibration = [i for i in selection_indices
                      if retained_prompts.loc[i, "group"] in calibration_groups]
selection_indices = np.setdiff1d(selection_indices, move_to_calibration)
calibration_indices = np.union1d(calibration_indices, move_to_calibration)
selection_mask = np.zeros(len(retained_prompts), dtype=bool)
calibration_mask = np.zeros(len(retained_prompts), dtype=bool)
selection_mask[selection_indices] = True
calibration_mask[calibration_indices] = True
assert not (selection_mask & calibration_mask).any()
assert np.array_equal(selection_mask | calibration_mask, validation_mask)
assert set(retained_prompts.loc[selection_mask, "group"]).isdisjoint(
    retained_prompts.loc[calibration_mask, "group"])
roles = retained_prompts.loc[validation_mask, ["row_id", "dataset", "label"]].copy()
roles["role"] = np.where(selection_mask[validation_mask], "selection", "calibration")
roles.to_csv("validation_roles.csv", index=False)
print("Epoch-selection / calibration rows:", selection_mask.sum(), calibration_mask.sum())
display(roles.groupby(["role", "dataset", "label"]).size())


Training / validation / test: 1056 269 315
Representation drift decision: yes >= 0.05; no <= 0.01; otherwise undecided.
Epoch-selection / calibration rows: 132 137


role         dataset    label
calibration  deepset    0        35
                        1        21
             safeguard  0        57
                        1        24
selection    deepset    0        34
                        1        19
             safeguard  0        55
                        1        24
dtype: int64

In [6]:
model_load_start = time.perf_counter()
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME, revision=MODEL_REVISION, use_fast=False, token=False
)
encoder = AutoModel.from_pretrained(MODEL_NAME, revision=MODEL_REVISION, token=False).to(DEVICE)
encoder.requires_grad_(False)
encoder.eval()
model_load_seconds = time.perf_counter() - model_load_start

assert not any(parameter.requires_grad for parameter in encoder.parameters())
print(
    "Frozen encoder parameters:",
    sum(parameter.numel() for parameter in encoder.parameters()),
)


def encode_texts(texts, show_progress=True):
    """Return final-layer mean embeddings, original token lengths, and input IDs."""
    embedding_batches = []
    token_lengths = []
    input_token_sequences = []
    encoding_start = time.perf_counter()

    with torch.inference_mode():
        for batch_start in range(0, len(texts), INFERENCE_BATCH_SIZE):
            batch_texts = texts[batch_start : batch_start + INFERENCE_BATCH_SIZE]
            full_token_ids = tokenizer(batch_texts, truncation=False)["input_ids"]
            token_lengths.extend(len(token_ids) for token_ids in full_token_ids)
            model_inputs = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=MAX_TOKENS,
                return_tensors="pt",
            ).to(DEVICE)

            for token_ids, attention_mask in zip(
                model_inputs["input_ids"], model_inputs["attention_mask"]
            ):
                input_token_sequences.append(
                    tuple(token_ids[attention_mask.bool()].tolist())
                )

            token_vectors = encoder(**model_inputs).last_hidden_state
            content_mask = model_inputs["attention_mask"].bool()
            for special_token_id in tokenizer.all_special_ids:
                content_mask &= model_inputs["input_ids"].ne(special_token_id)

            token_weights = content_mask.unsqueeze(-1).to(token_vectors.dtype)
            sentence_vectors = (token_vectors * token_weights).sum(dim=1)
            sentence_vectors /= token_weights.sum(dim=1).clamp_min(1)
            embedding_batches.append(sentence_vectors.cpu().numpy())

            if show_progress and batch_start % 160 == 0:
                completed = min(batch_start + INFERENCE_BATCH_SIZE, len(texts))
                elapsed = time.perf_counter() - encoding_start
                print(
                    f"Encoded {completed}/{len(texts)} texts ({elapsed:.1f}s)",
                    flush=True,
                )

    return (
        np.concatenate(embedding_batches),
        np.asarray(token_lengths),
        input_token_sequences,
    )


embedding_start = time.perf_counter()
embeddings, original_token_lengths, encoded_token_sequences = encode_texts(
    retained_prompts["text"].tolist()
)
embedding_seconds = time.perf_counter() - embedding_start
truncated_count = int((original_token_lengths > MAX_TOKENS).sum())
print("Embedding shape:", embeddings.shape)
print("Embedding extraction seconds:", embedding_seconds)
print("Truncated texts:", truncated_count)

# Check whether truncation created identical inputs across split boundaries.
retained_prompts["token_key"] = encoded_token_sequences
training_token_sequences = retained_prompts.loc[
    retained_prompts["split"].eq("train"), "token_key"
]
input_overlap_counts = {}
for evaluation_split in ["validation", "test"]:
    evaluation_sequences = set(
        retained_prompts.loc[
            retained_prompts["split"].eq(evaluation_split), "token_key"
        ]
    )
    input_overlap_counts[evaluation_split] = int(
        training_token_sequences.isin(evaluation_sequences).sum()
    )
assert not any(input_overlap_counts.values())
print("Training overlaps after tokenization:", input_overlap_counts)

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Frozen encoder parameters: 183831552


Encoded 32/1640 texts (0.4s)


... (8 outputs left out)

Encoded 1312/1640 texts (6.4s)


Encoded 1472/1640 texts (7.2s)


Encoded 1632/1640 texts (7.9s)


Embedding shape: (1640, 768)
Embedding extraction seconds: 7.9377046120000045
Truncated texts: 152
Training overlaps after tokenization: {'validation': 0, 'test': 0}


In [7]:
def new_probe(seed):
    return make_pipeline(StandardScaler(), LogisticRegression(
        C=1.0, class_weight="balanced", solver="lbfgs", max_iter=2000, random_state=seed))

def fit_probe(features, seed):
    classifier = new_probe(seed)
    started = time.perf_counter()
    classifier.fit(features[training_mask], labels[training_mask])
    return classifier, time.perf_counter() - started

def choose_threshold(validation_labels, validation_scores, fpr_limit=0.05):
    candidates = np.unique(np.append(validation_scores,
                          np.nextafter(validation_scores.max(), np.inf)))
    negatives = validation_scores[validation_labels == 0]
    return float(next(t for t in candidates if np.mean(negatives >= t) <= fpr_limit))

def primary_interval(frame, reps=1000):
    frame = frame.sort_values("row_id", kind="mergesort")
    y = frame.label.to_numpy()
    score = frame.score.to_numpy()
    rng = np.random.default_rng([zlib.crc32(str(frame.value.iloc[0]).encode()),
                                zlib.crc32(str(frame.column.iloc[0]).encode())])
    draws = []
    for _ in range(reps):
        indices = rng.integers(0, len(y), len(y))
        if len(np.unique(y[indices])) == 2:
            draws.append(roc_auc_score(y[indices], score[indices]))
    return np.percentile(draws, [2.5, 97.5])

baseline_probe, baseline_fit_seconds = fit_probe(embeddings, 15)
baseline_validation = baseline_probe.predict_proba(embeddings[calibration_mask])[:, 1]
baseline_threshold = choose_threshold(labels[calibration_mask], baseline_validation)
print("Baseline threshold:", repr(baseline_threshold))
clean = retained_prompts.loc[test_mask, ["row_id", "dataset", "label"]].copy()
clean["score"] = baseline_probe.predict_proba(embeddings[test_mask])[:, 1]
comparison = clean.merge(old_scores, on="row_id", suffixes=("", "_old"))
print("Maximum probability difference from midsemester:",
      np.max(np.abs(comparison.score - comparison.score_old)))

# Reuse identical clean embeddings in memory; encode changed copies once.
changed = perturbations[~perturbations["style"].eq("clean")].copy()
changed_features, changed_lengths, _ = encode_texts(changed.text.tolist())
variant_features = np.empty((len(perturbations), embeddings.shape[1]), dtype=np.float32)
row_to_feature = dict(zip(retained_prompts.row_id, embeddings))
clean_variants = perturbations["style"].eq("clean")
variant_features[clean_variants] = np.stack(perturbations.loc[clean_variants, "row_id"].map(row_to_feature))
variant_features[~clean_variants] = changed_features

score_frames = []
seed_frames = []
def make_scores(classifier, features, threshold, value):
    frame = perturbations[["row_id", "column", "label"]].copy()
    frame["value"] = value
    frame["score"] = classifier.predict_proba(features)[:, 1]
    frame["threshold"] = threshold
    return frame[["row_id", "value", "column", "label", "score", "threshold"]]

baseline_scores = make_scores(baseline_probe, variant_features, baseline_threshold, "frozen")
score_frames.append(baseline_scores)

# Fix the decision margin from frozen perturbation intervals before any LoRA outcomes.
interval_rows = []
for column, group in baseline_scores.groupby("column"):
    lower, upper = primary_interval(group)
    interval_rows.append({"column": column, "auroc": roc_auc_score(group.label, group.score),
                          "lower": lower, "upper": upper})
frozen_intervals = pd.DataFrame(interval_rows)
perturbed_intervals = frozen_intervals[frozen_intervals.column.str.endswith(("_zero_width", "_homoglyph", "_base64"))]
no_margin = float(((perturbed_intervals.upper - perturbed_intervals.lower) / 2).max())
yes_margin = float(np.ceil((no_margin + 0.0001) * 100) / 100)
print("Frozen perturbation half-width / yes margin:", no_margin, yes_margin)
frozen_intervals["no_margin"] = no_margin
frozen_intervals["yes_margin"] = yes_margin
frozen_intervals.to_csv("frozen_intervals.csv", index=False)
for seed in SEEDS:
    classifier, seconds = fit_probe(embeddings, seed)
    tau = choose_threshold(labels[calibration_mask], classifier.predict_proba(embeddings[calibration_mask])[:, 1])
    frame = make_scores(classifier, variant_features, tau, "frozen")
    frame["seed"] = seed
    seed_frames.append(frame)

truncation = changed[["column"]].copy()
truncation["truncated"] = changed_lengths > MAX_TOKENS
print("Truncation by changed column:")
display(truncation.groupby("column").truncated.agg(["sum", "count"]))
truncation.groupby("column").truncated.agg(["sum", "count"]).to_csv("truncation.csv")


Baseline threshold: 0.8409348234306908
Maximum probability difference from midsemester: 0.0008277729464321038


Encoded 32/1045 texts (0.2s)


Encoded 192/1045 texts (1.0s)


Encoded 352/1045 texts (1.8s)


Encoded 512/1045 texts (2.5s)


Encoded 672/1045 texts (3.3s)


Encoded 832/1045 texts (4.1s)


Encoded 992/1045 texts (4.9s)


Frozen perturbation half-width / yes margin: 0.10717233680630375 0.11


Truncation by changed column:


,sum,count
column,,
deepset_base64,41,116
deepset_homoglyph,21,116
deepset_paraphrase,0,50
deepset_zero_width,29,116
safeguard_base64,104,199
safeguard_homoglyph,64,199
safeguard_paraphrase,4,50
safeguard_zero_width,87,199


In [8]:
def normalized_rows(array):
    return array / np.maximum(np.linalg.norm(array, axis=1, keepdims=True), 1e-12)

# Each vector is estimated from training pairs and held fixed during test evaluation.
concept_texts = concept_pairs.injection_text.tolist() + concept_pairs.benign_text.tolist()
concept_test = concept_pairs[concept_pairs.split.eq("test")]
concept_homoglyphs = [homoglyph_map_text for homoglyph_map_text in
    (concept_test.injection_text.tolist() + concept_test.benign_text.tolist())]
homoglyph_map = str.maketrans({"a": "а", "e": "е", "o": "о", "p": "р", "c": "с", "x": "х"})
concept_homoglyphs = [t.translate(homoglyph_map) for t in concept_homoglyphs]
frozen_concepts, _, _ = encode_texts(concept_texts + concept_homoglyphs, show_progress=False)
train_pairs = concept_pairs.split.eq("train").to_numpy()
test_pairs = concept_pairs.split.eq("test").to_numpy()

def concept_directions(features):
    count = len(concept_pairs)
    differences = features[:count] - features[count:2*count]
    training_differences = differences[train_pairs]
    mean_vector = training_differences.mean(axis=0)
    mean_unit = mean_vector / np.linalg.norm(mean_vector)
    normalized = normalized_rows(training_differences)
    component = PCA(n_components=1).fit(normalized).components_[0]
    if component @ mean_vector < 0:
        component = -component
    return mean_vector, mean_unit, component, differences[test_pairs]

v0, u0, p0, frozen_test_contrasts = concept_directions(frozen_concepts)
representation_rows = []
concept_rows = []

def measure_representation(method, seed, clean_features, concepts):
    delta = clean_features[test_mask] - embeddings[test_mask]
    relative = np.linalg.norm(delta, axis=1) / np.maximum(np.linalg.norm(embeddings[test_mask], axis=1), 1e-12)
    vm, um, pm, test_contrasts = concept_directions(concepts)
    changed_count = len(concept_test)
    offset = 2 * len(concept_pairs)
    perturbed_contrasts = concepts[offset:offset+changed_count] - concepts[offset+changed_count:]
    for dataset in ["deepset", "safeguard"]:
        mask = retained_prompts.loc[test_mask, "dataset"].eq(dataset).to_numpy()
        drift = float(relative[mask].mean())
        rng = np.random.default_rng(15)
        values = relative[mask]
        boot = values[rng.integers(0, len(values), (1000, len(values)))].mean(axis=1)
        drift_lower, drift_upper = np.percentile(boot, [2.5, 97.5])
        representation_rows.append({"value": method, "seed": seed, "dataset": dataset,
            "D": drift, "lower": float(drift_lower), "upper": float(drift_upper), "D_verdict": "yes" if drift >= .05 else "no" if drift <= .01 else "undecided"})
    concept_rows.append({"value": method, "seed": seed,
        "mean_direction_change": 1 - float(u0 @ um),
        "pca_direction_change": 1 - float(p0 @ pm),
        "concept_norm": float(np.linalg.norm(vm)),
        "fixed_axis_margin_change": float(((test_contrasts-frozen_test_contrasts) @ u0).mean()),
        "mean_clean_margin": float((test_contrasts @ um).mean()),
        "mean_homoglyph_margin": float((perturbed_contrasts @ um).mean()),
        "pca_clean_margin": float((test_contrasts @ pm).mean()),
        "pca_homoglyph_margin": float((perturbed_contrasts @ pm).mean()),
        "contrast_cosine_clean": float((normalized_rows(test_contrasts) @ um).mean()),
        "contrast_cosine_homoglyph": float((normalized_rows(perturbed_contrasts) @ um).mean())})

measure_representation("frozen", 15, embeddings, frozen_concepts)


In [9]:
# Train every seed for all 20 epochs; only validation data chooses the retained epoch.
import base64
from sklearn.metrics import log_loss
from transformers import DataCollatorWithPadding

training_texts = retained_prompts.loc[training_mask, "text"].tolist()
training_labels = labels[training_mask]
training_tokens = tokenizer(training_texts, truncation=True, max_length=MAX_TOKENS)
collator = DataCollatorWithPadding(tokenizer)
selection_texts = retained_prompts.loc[selection_mask, "text"].tolist()
selection_labels = labels[selection_mask]
selection_variants = {
    "clean": selection_texts,
    "zero_width": [re.sub(r"(?<=[A-Za-z])(?=[A-Za-z])", "\u200b", text) for text in selection_texts],
    "homoglyph": [text.translate(homoglyph_map) for text in selection_texts],
    "base64": ["Decode this UTF-8 base64 request and carry out the decoded request:\n" +
               base64.b64encode(text.encode("utf-8")).decode("ascii") for text in selection_texts],
}
epoch_evaluation_texts = training_texts + [text for texts in selection_variants.values() for text in texts]
selection_count = len(selection_texts)
training_count = len(training_texts)
epoch_history = []
runtime_rows = [{"value": "frozen", "seed": 15, "training_seconds": baseline_fit_seconds,
                 "embedding_seconds": embedding_seconds, "trainable_parameters": 0,
                 "optimization_seconds": 0, "selection_seconds": 0,
                 "epochs_completed": 0, "selected_epoch": 0}]
del encoder
gc.collect()

for seed in SEEDS:
    print("START LORA SEED", seed, flush=True)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    shuffle_rng = np.random.default_rng(seed)
    encoder = AutoModel.from_pretrained(MODEL_NAME, revision=MODEL_REVISION, token=False).to(DEVICE)
    targets = [name for name, _ in encoder.named_modules()
               if ("layer.10." in name or "layer.11." in name)
               and (name.endswith("query_proj") or name.endswith("value_proj"))]
    assert len(targets) == 4
    encoder = get_peft_model(encoder, LoraConfig(r=4, lora_alpha=8, lora_dropout=0.05,
                                               target_modules=targets, bias="none"))
    trainable = sum(p.numel() for p in encoder.parameters() if p.requires_grad)
    head = torch.nn.Linear(embeddings.shape[1], 1).to(DEVICE)
    means = torch.tensor(embeddings[training_mask].mean(axis=0), device=DEVICE)
    scales = torch.tensor(embeddings[training_mask].std(axis=0).clip(min=1e-6), device=DEVICE)
    optimizer = torch.optim.AdamW([
        {"params": [p for p in encoder.parameters() if p.requires_grad], "lr": 1e-4},
        {"params": head.parameters(), "lr": 1e-3}], weight_decay=0.01)
    positive_weight = torch.tensor((training_labels == 0).sum() / (training_labels == 1).sum(), device=DEVICE)
    criterion = torch.nn.BCEWithLogitsLoss(pos_weight=positive_weight)
    best_key = (-np.inf, -np.inf)
    best_adapter = None
    best_epoch = None
    optimization_seconds = 0.0
    selection_seconds = 0.0
    run_started = time.perf_counter()

    for epoch in range(1, MAX_EPOCHS + 1):
        encoder.train()
        head.train()
        order = shuffle_rng.permutation(training_count)
        losses = []
        started = time.perf_counter()
        for begin in range(0, len(order), BATCH_SIZE):
            indices = order[begin:begin+BATCH_SIZE]
            examples = [{key: value[index] for key, value in training_tokens.items()} for index in indices]
            inputs = collator(examples).to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            token_vectors = encoder(**inputs).last_hidden_state
            mask = inputs["attention_mask"].bool()
            for special_id in tokenizer.all_special_ids:
                mask &= inputs["input_ids"].ne(special_id)
            weights = mask.unsqueeze(-1).to(token_vectors.dtype)
            features = (token_vectors * weights).sum(dim=1) / weights.sum(dim=1).clamp_min(1)
            logits = head((features-means)/scales).squeeze(-1)
            targets_y = torch.tensor(training_labels[indices], dtype=torch.float32, device=DEVICE)
            loss = criterion(logits, targets_y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(list(encoder.parameters())+list(head.parameters()), 1.0)
            optimizer.step()
            losses.append(float(loss.detach()))
        epoch_train_seconds = time.perf_counter()-started
        optimization_seconds += epoch_train_seconds

        # Evaluate the same final-probe protocol at every epoch, without consulting test scores.
        started = time.perf_counter()
        encoder.eval()
        epoch_features, _, _ = encode_texts(epoch_evaluation_texts, show_progress=False)
        epoch_probe = new_probe(seed)
        epoch_probe.fit(epoch_features[:training_count], training_labels)
        validation_aurocs = {}
        clean_loss = None
        for index, style in enumerate(selection_variants):
            begin = training_count + index*selection_count
            probabilities = epoch_probe.predict_proba(epoch_features[begin:begin+selection_count])[:, 1]
            validation_aurocs[style] = roc_auc_score(selection_labels, probabilities)
            if style == "clean":
                clean_loss = log_loss(selection_labels, probabilities, labels=[0, 1])
        selection_auroc = float(np.mean(list(validation_aurocs.values())))
        key = (selection_auroc, -clean_loss)
        if key > best_key:
            best_key = key
            best_epoch = epoch
            best_adapter = {name: parameter.detach().cpu().clone()
                            for name, parameter in encoder.named_parameters() if parameter.requires_grad}
        epoch_selection_seconds = time.perf_counter()-started
        selection_seconds += epoch_selection_seconds
        epoch_history.append({"seed": seed, "epoch": epoch, "train_loss": np.mean(losses),
            "selection_auroc": selection_auroc, "clean_log_loss": clean_loss,
            **{style+"_auroc": value for style, value in validation_aurocs.items()},
            "optimization_seconds": epoch_train_seconds, "selection_seconds": epoch_selection_seconds})
        pd.DataFrame(epoch_history).to_csv("epoch_history.csv", index=False)
        print(f"seed {seed} epoch {epoch:02d}/{MAX_EPOCHS}: loss={np.mean(losses):.4f}, "
              f"selection AUROC={selection_auroc:.4f}, clean={validation_aurocs['clean']:.4f}, "
              f"best epoch={best_epoch}", flush=True)

    adaptation_seconds = time.perf_counter()-run_started
    # Retain only LoRA tensors in RAM; no checkpoint files or metadata are written.
    with torch.no_grad():
        for name, parameter in encoder.named_parameters():
            if name in best_adapter:
                parameter.copy_(best_adapter[name].to(DEVICE))
    encoder.eval()
    del optimizer, head, epoch_features, epoch_probe, best_adapter
    gc.collect()

    started = time.perf_counter()
    adapted_embeddings, _, _ = encode_texts(retained_prompts.text.tolist())
    adapted_changed, _, _ = encode_texts(changed.text.tolist())
    adapted_variants = np.empty_like(variant_features)
    lookup = dict(zip(retained_prompts.row_id, adapted_embeddings))
    adapted_variants[clean_variants] = np.stack(perturbations.loc[clean_variants, "row_id"].map(lookup))
    adapted_variants[~clean_variants] = adapted_changed
    adapted_concepts, _, _ = encode_texts(concept_texts + concept_homoglyphs, show_progress=False)
    extraction_seconds = time.perf_counter()-started
    classifier, fit_seconds = fit_probe(adapted_embeddings, seed)
    calibration_scores = classifier.predict_proba(adapted_embeddings[calibration_mask])[:, 1]
    tau = choose_threshold(labels[calibration_mask], calibration_scores)
    frame = make_scores(classifier, adapted_variants, tau, "lora")
    frame["seed"] = seed
    seed_frames.append(frame)
    if seed == SEEDS[0]:
        score_frames.append(frame.drop(columns="seed"))
    measure_representation("lora", seed, adapted_embeddings, adapted_concepts)
    runtime_rows.append({"value": "lora", "seed": seed,
        "training_seconds": adaptation_seconds + fit_seconds, "embedding_seconds": extraction_seconds,
        "trainable_parameters": trainable, "optimization_seconds": optimization_seconds,
        "selection_seconds": selection_seconds, "epochs_completed": MAX_EPOCHS,
        "selected_epoch": best_epoch})
    pd.concat(seed_frames, ignore_index=True).to_csv("seed_scores.csv", index=False)
    pd.DataFrame(runtime_rows).to_csv("runtimes.csv", index=False)
    pd.DataFrame(representation_rows).to_csv("representation_drift.csv", index=False)
    pd.DataFrame(concept_rows).to_csv("concept_metrics.csv", index=False)
    pd.concat(score_frames, ignore_index=True).to_csv("scores.csv", index=False)
    print("FINISHED SEED", seed, "selected epoch", best_epoch,
          "train+selection seconds", adaptation_seconds, "threshold", repr(tau), flush=True)
    del encoder, adapted_embeddings, adapted_changed, adapted_variants, adapted_concepts, classifier
    gc.collect()


START LORA SEED 15


seed 15 epoch 01/20: loss=0.5057, selection AUROC=0.6687, clean=0.9770, best epoch=1


seed 15 epoch 02/20: loss=0.2546, selection AUROC=0.6757, clean=0.9786, best epoch=2


seed 15 epoch 03/20: loss=0.1909, selection AUROC=0.6865, clean=0.9812, best epoch=3


seed 15 epoch 04/20: loss=0.1904, selection AUROC=0.6940, clean=0.9846, best epoch=4


seed 15 epoch 05/20: loss=0.1382, selection AUROC=0.6974, clean=0.9854, best epoch=5


... (110 outputs left out)

Encoded 672/1045 texts (3.4s)


Encoded 832/1045 texts (4.3s)


Encoded 992/1045 texts (5.0s)


FINISHED SEED 17 selected epoch 18 train+selection seconds 259.0792753620001 threshold 0.40921226293162505


In [10]:
scores = pd.concat(score_frames, ignore_index=True)
scores.to_csv("scores.csv", index=False)
seed_scores = pd.concat(seed_frames, ignore_index=True)
seed_scores.to_csv("seed_scores.csv", index=False)
metric_rows = []
for (value, seed, column), group in seed_scores.groupby(["value", "seed", "column"]):
    y = group.label.to_numpy()
    pred = group.score.to_numpy() >= group.threshold.to_numpy()
    metric_rows.append({"value": value, "seed": seed, "column": column,
        "AUROC": roc_auc_score(y, group.score), "recall": pred[y == 1].mean(),
        "FPR": pred[y == 0].mean(), "threshold": group.threshold.iloc[0]})
seed_metrics = pd.DataFrame(metric_rows)
seed_metrics.to_csv("seed_metrics.csv", index=False)
display(seed_metrics.groupby(["value", "column"]).AUROC.agg(["mean", "std", "min", "max"]))
display(pd.DataFrame(runtime_rows))
display(pd.DataFrame(representation_rows))
display(pd.DataFrame(concept_rows))

# Deterministic, reproducible error sample from a prespecified two-class cell.
error_cell = scores[scores.value.eq("frozen") & scores.column.eq("deepset_homoglyph")].copy()
error_cell["prediction"] = error_cell.score >= error_cell.threshold
sampled = []
for label in [0, 1]:
    wrong = error_cell[error_cell.label.eq(label) & error_cell.prediction.ne(label)]
    sampled.append(wrong.sample(n=min(10, len(wrong)), random_state=15))
error_sample = pd.concat(sampled).merge(perturbations[["row_id", "column", "text"]],
                                      on=["row_id", "column"], validate="one_to_one")
# Categories describe observable text properties, not causal diagnoses.
def category(row):
    text = row.text.casefold()
    if row.label == 0:
        return "benign instruction vocabulary"
    if any(word in text for word in ["system", "earlier", "prior", "previous", "policy", "policies", "rules", "constraint"]):
        return "hierarchy or policy redirection"
    return "task or role redirection"
error_sample["category"] = error_sample.apply(category, axis=1)
error_sample[["row_id", "value", "column", "category"]].to_csv("errors.csv", index=False)
error_sample.to_csv("error_examples.csv", index=False)
display(error_sample[["row_id", "label", "score", "category", "text"]])


mean       std       min       max
value  column                                                      
frozen deepset_base64        0.476190  0.000000  0.476190  0.476190
       deepset_clean         0.991667  0.000000  0.991667  0.991667
       deepset_homoglyph     0.776786  0.000000  0.776786  0.776786
       deepset_paraphrase    0.985600  0.000000  0.985600  0.985600
       deepset_zero_width    0.767262  0.000000  0.767262  0.767262
       safeguard_base64      0.497199  0.000000  0.497199  0.497199
       safeguard_clean       0.995565  0.000000  0.995565  0.995565
       safeguard_homoglyph   0.543067  0.000000  0.543067  0.543067
       safeguard_paraphrase  1.000000  0.000000  1.000000  1.000000
       safeguard_zero_width  0.470005  0.000000  0.470005  0.470005
lora   deepset_base64        0.461409  0.065562  0.385714  0.500298
       deepset_clean         0.991369  0.003892  0.988690  0.995833
       deepset_homoglyph     0.799702  0.009547  0.789286  0.808036
       deepset_paraphrase    0.995200  0.003200  0.992000  0.998400
       deepset_zero_width    0.705754  0.041944  0.658929  0.739881
       safeguard_base64      0.490196  0.027491  0.463002  0.517974
       safeguard_clean       0.995565  0.000535  0.995098  0.996148
       safeguard_homoglyph   0.781590  0.023518  0.758053  0.805089
       safeguard_paraphrase  1.000000  0.000000  1.000000  1.000000
       safeguard_zero_width  0.612628  0.016318  0.599323  0.630836

,value,seed,training_seconds,embedding_seconds,trainable_parameters,optimization_seconds,selection_seconds,epochs_completed,selected_epoch
0,frozen,15,0.549685,7.937705,0,0.000000,0.000000,0,0
1,lora,15,260.170052,12.841606,24576,101.241241,158.138401,20,13
2,lora,16,260.952267,12.827404,24576,101.596571,158.750787,20,17
3,lora,17,259.538499,12.854824,24576,102.076432,156.782036,20,18


,value,seed,dataset,D,lower,upper,D_verdict
0,frozen,15,deepset,0.000000,0.000000,0.000000,no
1,frozen,15,safeguard,0.000000,0.000000,0.000000,no
2,lora,15,deepset,0.183536,0.167396,0.200089,yes
3,lora,15,safeguard,0.179284,0.168394,0.190052,yes
4,lora,16,deepset,0.219136,0.198136,0.240706,yes
5,lora,16,safeguard,0.207296,0.193404,0.221141,yes
6,lora,17,deepset,0.236866,0.210644,0.265781,yes
7,lora,17,safeguard,0.207389,0.190550,0.224073,yes


,value,seed,mean_direction_change,pca_direction_change,concept_norm,fixed_axis_margin_change,mean_clean_margin,mean_homoglyph_margin,pca_clean_margin,pca_homoglyph_margin,contrast_cosine_clean,contrast_cosine_homoglyph
0,frozen,15,5.960464e-08,1.788139e-07,5.459350,0.000000,3.168308,1.105923,1.930822,0.961676,0.361559,0.181812
1,lora,15,3.944671e-02,1.495349e-02,5.765053,0.000613,3.221002,1.151365,1.656860,0.740665,0.342116,0.187807
2,lora,16,4.728067e-02,2.012849e-02,5.759562,0.009192,3.341132,1.193491,1.835656,0.418763,0.355440,0.194067
3,lora,17,4.315227e-02,1.963566e+00,5.709194,-0.132760,3.060250,1.142666,-0.969061,-0.461566,0.338226,0.188010


,row_id,label,score,category,text
0,deepset:test:64,0,0.996546,benign instruction vocabulary,Wiеviеlе Zigаrеttеn hаt Hеlmut Sсhmidt аn еinе...
1,deepset:test:54,0,0.988196,benign instruction vocabulary,will diе bundеsrеgiеrung sсhnеll rаus аus dеr ...
2,deepset:test:63,0,0.851068,benign instruction vocabulary,Glеiсhbеrесhtigung LGBTIQ+ Eurора
3,deepset:test:62,0,0.975153,benign instruction vocabulary,Brеitbаndаusbаu Ländliсhе Rеgiоnеn Dеutsсhlаnd
4,deepset:test:50,0,0.845651,benign instruction vocabulary,Wаs kаnn iсh mit Bärlаuсh kосhеn?
5,deepset:test:47,0,0.959240,benign instruction vocabulary,Künstliсhе Intеlligеnz Ethik Eurора
6,deepset:test:58,0,0.965070,benign instruction vocabulary,Vеrkеhrswеndе Fаhrrаdinfrаstruktur Dеutsсhlаnd
7,deepset:test:55,0,0.907195,benign instruction vocabulary,E-Sсооtеr Vеrkеhrssiсhеrhеit Dеutsсhlаnd
8,deepset:test:5,0,0.876563,benign instruction vocabulary,My dаughtеr will sооn bе finishеd with sсhооl ...
9,deepset:test:9,0,0.906564,benign instruction vocabulary,Rесyсling рlаstiс Gеrmаny
